# Import Packages and Functions

## Code: Loading Packages and Functions
Run the code below to load all required packages and helper functions. You only need to run this section once.

In [ ]:
import pandas as pd # used to handle csv- or excel files as a dataframe (table object)
import numpy as np # used for basic mathematical operations
import matplotlib.pyplot as plt # package for basic data plotting
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import seaborn as sns # additional package with more plotting options based on pyplot


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
import os 
os.environ["OMP_NUM_THREADS"] = "1"

# from here code for more complexe plotting functions
def _resolve_column(df: pd.DataFrame, col):
    """Resolve a column specifier for flat or MultiIndex columns.
    - col can be None, a tuple (MultiIndex), or a string.
    - Exact matches are preferred. If not found and df has MultiIndex,
      try to find columns where any level equals the string.
    """
    if left_plot not in df.columns or right_plot not in df.columns:
        raise ValueError(f"KMeans oder GMM Label sind nicht im DataFrame vorhanden. Bitte erst Kmeans- und GMM-Clustering durchführen!")
    if col is None:
        return None, None  # (series, display_name)
    # tuple (explicit MultiIndex)
    if isinstance(col, tuple):
        if col in df.columns:
            return df[col], " - ".join(map(str, col))
        raise KeyError(f"Column tuple {col} not found in DataFrame columns.")
    # string case
    # direct exact match (flat columns)
    if col in df.columns:
        return df[col], str(col)
    # if MultiIndex, try exact match on any level
    if isinstance(df.columns, pd.MultiIndex):
        # exact level match
        matches = [c for c in df.columns if any(str(level) == col for level in c)]
        if len(matches) == 1:
            return df[matches[0]], " - ".join(map(str, matches[0]))
        if len(matches) > 1:
            # prefer a match where top-level equals col
            top_matches = [c for c in matches if str(c[0]) == col]
            chosen = top_matches[0] if top_matches else matches[0]
            print(f"Multiple columns match '{col}', using {chosen}.")
            return df[chosen], " - ".join(map(str, chosen))
        # try substring match (e.g., user passes 'PC 1' and column is ('PCA','PC 1'))
        substr_matches = [c for c in df.columns if any(col in str(level) for level in c)]
        if len(substr_matches) >= 1:
            chosen = substr_matches[0]
            print(f"No exact match for '{col}', using first substring match {chosen}.")
            return df[chosen], " - ".join(map(str, chosen))
    # fallback: try substring in flat columns
    flat_sub = [c for c in df.columns if col in str(c)]
    if len(flat_sub) >= 1:
        chosen = flat_sub[0]
        print(f"No exact match for '{col}', using first flat substring match {chosen}.")
        return df[chosen], str(chosen)
    raise KeyError(f"Column '{col}' not found in DataFrame columns.")

def plot_dataframe(df: pd.DataFrame,
                   x,
                   y,
                   z=None,
                   hue=None,
                   figsize=(9, 6),
                   palette=None,
                   point_size=40,
                   alpha=0.9,
                   cmap="viridis",
                   title=None):
    """
    Flexible plotting for DataFrame: automatic 2D/3D scatter depending on z.
    x, y, z, hue can be strings or tuples (for MultiIndex). If z is None -> 2D.
    Returns (fig, ax).
    """
    if left_plot not in df.columns or right_plot not in df.columns:
        raise ValueError(f"KMeans oder GMM Label sind nicht im DataFrame vorhanden. Bitte erst Kmeans- und GMM-Clustering durchführen!")
    # Resolve columns and display names
    X, x_label = _resolve_column(df, x)
    Y, y_label = _resolve_column(df, y)
    Z, z_label = _resolve_column(df, z) if z is not None else (None, None)
    H, h_label = _resolve_column(df, hue) if hue is not None else (None, None)

    if X is None or Y is None:
        raise ValueError("x and y must be provided and resolvable to DataFrame columns.")

    x_vals = X.values
    y_vals = Y.values
    z_vals = Z.values if Z is not None else None
    hue_vals = H.values if H is not None else None

    is_3d = z_vals is not None

    fig = plt.figure(figsize=figsize)
    if is_3d:
        ax = fig.add_subplot(111, projection='3d')
    else:
        ax = fig.add_subplot(111)

    # No hue: single color
    if hue_vals is None:
        color = sns.color_palette()[0]
        if is_3d:
            sc = ax.scatter(x_vals, y_vals, z_vals, s=point_size, alpha=alpha, color=color)
        else:
            sc = ax.scatter(x_vals, y_vals, s=point_size, alpha=alpha, color=color)
    else:
        # Determine whether hue is numeric or categorical.
        # If dtype is object or string-like -> categorical.
        if pd.api.types.is_numeric_dtype(H):
            # continuous hue
            if is_3d:
                # 3D scatter with continuous color: map to RGBA
                norm = plt.Normalize(np.nanmin(hue_vals), np.nanmax(hue_vals))
                cmap_obj = plt.get_cmap(cmap)
                colors = cmap_obj(norm(hue_vals))
                sc = ax.scatter(x_vals, y_vals, z_vals, c=colors, s=point_size, alpha=alpha)
                # create a ScalarMappable for colorbar
                mappable = plt.cm.ScalarMappable(norm=norm, cmap=cmap_obj)
                mappable.set_array(hue_vals)
                cbar = fig.colorbar(mappable, ax=ax, pad=0.1)
                cbar.set_label(h_label or str(hue))
            else:
                sc = ax.scatter(x_vals, y_vals, c=hue_vals, cmap=cmap, s=point_size, alpha=alpha)
                cbar = fig.colorbar(sc, ax=ax, pad=0.1)
                cbar.set_label(h_label or str(hue))
        else:
            # categorical hue (strings or objects)
            categories, uniques = pd.factorize(hue_vals)
            n_cats = len(uniques)
            if palette is None:
                palette = sns.color_palette(n_colors=n_cats)
            colors = [palette[i % len(palette)] for i in categories]
            if is_3d:
                sc = ax.scatter(x_vals, y_vals, z_vals, c=colors, s=point_size, alpha=alpha)
            else:
                sc = ax.scatter(x_vals, y_vals, c=colors, s=point_size, alpha=alpha)
            # legend
            handles = []
            for i, lab in enumerate(uniques):
                handles.append(plt.Line2D([], [], marker='o', color=palette[i % len(palette)], linestyle='', markersize=6))
            ax.legend(handles, uniques, title=(h_label or str(hue)), bbox_to_anchor=(1.05, 1), loc='upper left')

    # Labels and title
    ax.set_xlabel(x_label or str(x))
    ax.set_ylabel(y_label or str(y))
    if is_3d:
        ax.set_zlabel(z_label or str(z))
    if title:
        ax.set_title(title)

    plt.tight_layout()
    plt.show()
    return fig, ax


# Data Import

The first step is to import the data. We use the "pandas" library for this. Pandas can read a wide variety of file formats including CSV, Excel, and even PNG. For large datasets, lightweight formats like CSV are preferred over Excel, since reading Excel files can be significantly slower.

## Code: Importing Data

There are two datasets available in the folder "Data/CSV/".

**OriginalData.csv** contains basic nanoindentation results (hardness, elastic modulus, stiffness) for each indent.

**DataWithAdditional_Infos.csv** contains the same data plus additional information describing the mathematical fit of the loading and unloading curves, as well as the trends of hardness and modulus as a function of indentation depth. These derived quantities were obtained by curve fitting the raw data and can add valuable features to the dataset. To start, load "OriginalData.csv" to get a clear overview of the data structure.

In [ ]:
#=== change parameter from here ===#

file_name:str = "OriginalData.csv" #<-- "OriginalData.csv" or "DataWithAdditional_Infos.csv"

#=== dont change paramter from here ===#

file_path = rf"Data/CSV/{file_name}"

# creating data frame from nanoindendation data
df = pd.read_csv(file_path, # path pointing to the file to be importe d 
                header = [0,1], # used because the dataframe has multi-index column names "("MODULUS GPa","mean") & ("MODULLUS GPa", "std")" as example
                sep = ";", # defines the separator used in csv file to separate different columns 
                decimal = "," #telling pandas the instad of "." a comma "," is used as decimal point (german number format)
                )
# Set Indent column as Index
df.index = df[("Unnamed: 0_level_0","Unnamed: 0_level_1")]
df = df.drop(("Unnamed: 0_level_0","Unnamed: 0_level_1"), axis=1)

#visualization of the data structure
display(df)

# Data Visualization
In the imported data, each indent has two sets of x- and y-coordinates: **"absolute"** and **"real"**. The nominal spacing between indents was set to 4 µm before the measurement. However, post-measurement mapping with the atomic force microscope (AFM) reveals that the actual spacing — especially at small step sizes — **can deviate significantly from the nominal value**.

The **"absolute" coordinates** are the idealized grid positions from the nanoindentation instrument, based on the planned step size. The **"real" coordinates** were determined after the fact using AFM imaging and reflect where the indents actually landed.

If you want to overlay multiple datasets — for example, nanoindentation data with AFM images (or with other techniques such as EBSD) — you need to **incorporate the real indent positions into the dataset**. Using image processing and appropriate scaling (e.g., to a 50 µm × 50 µm field of view), the true indent centers can be extracted.

## Code: Hardness and Elastic Modulus Mapping
Modify the adjustable parameters below and observe how they affect the plot.

In [ ]:
#=== change parameter from here ===#
#marker options
marker_shape = "s" #<-- "s" for squares, ">" for triangles
marker_size = 85 # setting the size of the markers
marker_tranparency = 0.75 # setting the transparency of the markers

#choosing between ideal indent position and real indent positions
indent_position = "real" # <-- "real" or "ideal"

#change the colormap of the plots
colormap = "crest_r" #<-- "viridis", "cividis", "inferno" ,"magma", "plasma", "rocket", "flare", "crest", "copper"
#=========================#

#=== dont change paramter from here ===#
#image size
image_width_cm = 15 #change value to alter the image size
image_height_cm = 15 # change value to alter the image size

#Atomic Force Microscopy Mapping of Indentationmapping
background_image = plt.imread("Data/Images/BackGround.png")
dx,dy = -2.5,-3.2 #parameter for adjusting the background image
range_um = 50 # parameter for adjusting the background image

# automatical column selection for indent position
if indent_position == "real":
    x = ("x","real") # defining x axis
    y = ("y","real") # defining y axis
elif indent_position =="ideal":
    x = ("x","absolut") # defining x axis
    y = ("y","absolut") # defining x axis

#creating a figure object
fig, ax = plt.subplots(nrows = 1,
                       ncols = 2, #3 images next to each other
                       sharey=True)


fig.set_dpi(600) # increasing the resolution of the plot
fig.set_size_inches(image_width_cm/2.5,image_height_cm/2.54) #calcuating image size

# hardness plot 
sns.scatterplot(data = df,
                x = x,
                y = y,
                hue = ("HARDNESS GPa","mean"),
                ax = ax[0], #left plot
                marker = marker_shape, #square marker
                palette = sns.color_palette(colormap,as_cmap = True), # defining the color plaette
                s = marker_size, # marker size
                edgecolor = None, # remove the outline of the markers
                alpha = marker_tranparency , # transparence of the markers infill
               )

if indent_position == "real":
    ax[0].imshow(background_image,
                 extent=[dx,range_um+dx,dy,range_um+dy],
                 aspect='equal',
                 zorder=-1,
                 origin = "upper"
                )

#modulus plot
sns.scatterplot(data = df,
                x = x,
                y = y,
                hue = ("MODULUS GPa","mean"),
                ax = ax[1], #middle plot
                marker = marker_shape, #square marker
                palette = sns.color_palette(colormap,as_cmap = True), # defining the color plaette
                s = marker_size, # marker size
                edgecolor = None, # remove the outline of the markers
                alpha = marker_tranparency , # transparence of the markers infill
                )

if indent_position == "real":
    ax[1].imshow(background_image,
                 extent=[dx,range_um+dx,dy,range_um+dy],
                 aspect='equal',
                 zorder=-1,
                 origin = "upper"
                )

for item in ax:
    item.set_aspect("equal") # ensure äquidistance on x and y axis
    sns.move_legend(loc = "lower center", bbox_to_anchor = (0.5,1), obj = item) # move legend above the corresponding plots


## Code: Cluster Mapping

Use the code below to create cluster maps.  
Unlike the hardness and elastic modulus maps, the indents are now color-coded by their **cluster label**. This produces a spatial map that highlights regions with similar mechanical properties, making it easy to spot patterns or distinct material zones.

You can also use this code to map continuous mechanical properties (just like the hardness map). To do so, change the `hue` parameter to a different column in your DataFrame and switch the colormap to `"crest_r"` or another sequential colormap.

In [ ]:
#=== change parameter from here ===#
#marker options
marker_shape = "s" #<-- "s" for squares, ">" for triangles
marker_size = 85 # setting the size of the markers
marker_tranparency = 0.75 # setting the transparency of the markers

#choosing between ideal indent position and real indent positions
indent_position = "real" # <-- "real" or "ideal"

#change the colormap of the plots
colormap = "tab10" #<-- "tab10", "tab20", "colorblind" 

left_plot = ("GMM","Label")
right_plot = ("KMeans","Label")
#=========================#

#=== dont change paramter from here ===#
if left_plot not in df.columns or right_plot not in df.columns:
    raise ValueError(f"KMeans oder GMM Label sind nicht im DataFrame vorhanden. Bitte erst Kmeans- und GMM-Clustering durchführen!")
#image size
image_width_cm = 15 #change value to alter the image size
image_height_cm = 15 # change value to alter the image size

#Atomic Force Microscopy Mapping of Indentationmapping
background_image = plt.imread("Data/Images/BackGround.png")
dx,dy = -2.5,-3.2 #parameter for adjusting the background image
range_um = 50 # parameter for adjusting the background image

# automatical column selection for indent position
if indent_position == "real":
    x = ("x","real") # defining x axis
    y = ("y","real") # defining y axis
elif indent_position =="ideal":
    x = ("x","absolut") # defining x axis
    y = ("y","absolut") # defining x axis

#creating a figure object
fig, ax = plt.subplots(nrows = 1,
                       ncols = 2, 
                       sharey=True)


fig.set_dpi(600) # increasing the resolution of the plot
fig.set_size_inches(image_width_cm/2.5,image_height_cm/2.54) #calcuating image size


sns.scatterplot(data = df,
                x = x,
                y = y,
                hue = left_plot,
                ax = ax[0],
                marker = marker_shape, #square marker
                palette = sns.color_palette(colormap)[2:], # defining the color plaette
                s = marker_size, # marker size
                edgecolor = None, # remove the outline of the markers
                alpha = marker_tranparency , # transparence of the markers infill
               )

if indent_position == "real":
    ax[0].imshow(background_image,
                 extent=[dx,range_um+dx,dy,range_um+dy],
                 aspect='equal',
                 zorder=-1,
                 origin = "upper"
                )

sns.scatterplot(data = df,
                x = x,
                y = y,
                hue = right_plot,
                ax = ax[1],
                marker = marker_shape, #square marker
                palette = sns.color_palette(colormap)[2:], # defining the color plaette
                s = marker_size, # marker size
                edgecolor = None, # remove the outline of the markers
                alpha = marker_tranparency , # transparence of the markers infill
               )

if indent_position == "real":
    ax[1].imshow(background_image,
                 extent=[dx,range_um+dx,dy,range_um+dy],
                 aspect='equal',
                 zorder=-1,
                 origin = "upper"
                )

for item in ax:
    item.set_aspect("equal") # ensure äquidistance on x and y axis
    sns.move_legend(loc = "lower center", bbox_to_anchor = (0.5,1), obj = item) # move legend above the corresponding plots


## Code: Exploring Cluster Visualizations

Use the code below to visualize the clusters in **feature space** or **PCA space**.  
Choose the columns for **x**, **y**, and **z** to define which quantities are plotted against each other.  
The **hue** parameter controls how the data points are colored.

The cluster labels work especially well here:  
- **("GMM", "Label")** or  
- **("KMeans", "Label")**  

but you can also use continuous (non-categorical) columns to explore additional relationships in the data.

For a **2D plot**, set:
z = None

In [ ]:
#=== change from here ===#
plot_dataframe(df, #<-- keep it 
               x=('PCA','PC1'), # choose x-axis
               y=("PCA",'PC2'), # choose y-axis
               z = None, # choose z-axis or set it to None for 2D plot
               hue=('KMeans','Label') # 
              ) 

plot_dataframe(df, #<-- keep it 
               x=('PCA','PC1'), # choose x-axis
               y=("PCA",'PC2'), # choose y-axis
               z = None, # choose z-axis or set it to None for 2D plot
               hue=('GMM','Label') # 
              ) 
#=== dont change from here ===#



# Principal Component Analysis (PCA)

## Choosing the Right Features

The parameter `n_components` controls how many principal components the dataset is reduced to. Typically 2 or 3 components are chosen, since they can be easily visualized.

PCA is a **dimensionality reduction** technique. When your dataset has many features (e.g., 5 features = 5 dimensions), PCA projects it down to fewer dimensions while preserving as much of the overall variance — and therefore the underlying structure — as possible. The goal is to capture the most important patterns in the data while discarding noise and redundancy.

## How to Interpret PCA Results

PCA transforms a high-dimensional dataset into a smaller set of new variables called **principal components (PCs)**. Each PC is a linear combination of the original features, and the components are ordered so that the first one captures the most variance, the second captures the next most, and so on.

A key metric is the **explained variance ratio**. It tells you what fraction of the total variance in the data each principal component accounts for. For example, if PC1 and PC2 together explain 85% of the variance, they capture the bulk of the information in the dataset. In that case, a 2D plot of those two components is usually a good representation for further analysis.

In a **PCA scatter plot**, each data point (here, each indent) is plotted according to its coordinates in the principal component space. Points that are close together have similar properties. Visible groupings suggest similar material behavior, while outliers may indicate anomalous or faulty measurements.

The **loadings** tell you how much each original feature contributes to a given principal component. This is important for physical interpretation. For example, if PC1 has large loadings for both hardness and elastic modulus, it likely captures the overall stiffness of the material. Another component might load heavily on the scatter (standard deviation) of your measurements, reflecting how homogeneous or heterogeneous the material is at that location. Features that contribute strongly to a component will have large positive or large negative loading values. It is perfectly normal for a feature to contribute strongly to one PC but not to others. However, if a feature has near-zero loadings on **all** principal components, it is not contributing useful information to the analysis and should be removed from the feature set.

By examining both the explained variance and the loadings, you can assess **which features drive the structure of your data** and make informed decisions about feature selection for clustering.

## Why Do We Need to Scale the Data?

Before running PCA or k-means, it is important to **standardize** the data — that is, to rescale all features so they are on a comparable numerical range.

The reason is that both PCA and k-means rely on **distance calculations in feature space**. If one feature (e.g., elastic modulus in GPa) has much larger numerical values than another (e.g., hardness in GPa), it will **dominate the distance metric** and therefore dominate the analysis — even if it is not inherently more important.

**Standard scaling** (e.g., using scikit-learn's `StandardScaler`) transforms each feature to have a **mean of 0 and a standard deviation of 1**. This ensures that all features **contribute on an equal footing**, regardless of their original units or magnitudes.

---

## Code: Running PCA  
Experiment with different values for `n_components` (e.g., 2 and 3) and try adding additional features to the `features` DataFrame that might be relevant for clustering — for example, the standard deviation of hardness, elastic modulus, or S²/P. If needed, scroll back up to review the column names in the original DataFrame.

In [ ]:
#=== change parameter ===#
#selecting important columns for clustering
features = df[[
                ("HARDNESS GPa","mean"),
                ("MODULUS GPa","mean"),
                ("S2overP","mean")
                ]].copy()

n_components = 2 #<-- determination of the dimension of the data space
#=========================#

#=== dont change paramter from here ===#

#deleting indents with not data (Indent 15, Indent 67) 
features.dropna(inplace = True)

# scale data very important for correct results!
scaler = StandardScaler() 
X_scaled = scaler.fit_transform(features)

# PCA on scaled data
pca = PCA(n_components=n_components) 
X_pca = pca.fit_transform(X_scaled)

fig = plt.figure(figsize=(8, 6)) 
if X_pca.shape[1] == 3:
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], c='steelblue', s=40)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_zlabel('PC3')
    ax.set_title('PCA Scatterplot (3D)')
    plt.show()
    print("Erklärte Varianzanteile:", pca.explained_variance_ratio_)
    loadings = pd.DataFrame(pca.components_.T, columns=['PCA1', 'PCA2',"PCA3"], index=features.columns ) 
    print(loadings)

else:
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c='steelblue', s=40)
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.title('PCA Scatterplot (2D)')
    plt.show()
    print("Erklärte Varianzanteile:", pca.explained_variance_ratio_)
    loadings = pd.DataFrame(pca.components_.T, columns=['PC 1', 'PC 2'], index=features.columns ) 
    print(loadings)

# deleting PCA columns in orignial data dataframe if present 
if 'PCA' in df.columns.get_level_values(0):
    df = df.drop(columns='PCA', level=0)

# make dataframe for PCA1 and PCA2 and maybe PCA3 axis
pca_columns = [f'PC{i+1}' for i in range(X_pca.shape[1])]
pca_df = pd.DataFrame(X_pca, columns=pca_columns, index=features.index)

# transform to multi index column ("PCA","PC 1),("PC","PC 2"),...
pca_df.columns = pd.MultiIndex.from_product([['PCA'], pca_columns])

# join pca_df with orignal DataFrame df by Index "Indent Nr"
df = df.join(pca_df)

#disply orignal DataFrame with PCA columns
display(df)

# Clustering with k-means

## What Is k-means and Why Is It Useful for Nanoindentation?

k-means is one of the most widely used **clustering algorithms**. It automatically partitions data points into groups (called *clusters*) — **without requiring any predefined labels**. This makes k-means a classic example of **unsupervised learning**: the algorithm discovers structure in the data on its own.

The algorithm works as follows:

1. **Initialize** `k` cluster centroids at random positions.
2. **Assign** each data point to the nearest centroid.
3. **Update** each centroid to be the mean of all points assigned to it.
4. **Repeat** steps 2 and 3 until the assignments stabilize (converge) or a maximum number of iterations is reached.

In nanoindentation, we measure mechanical properties such as **hardness**, **elastic modulus**, and **S²/P** at a large number of locations. These values vary depending on the material phase, microstructure, or surface condition. k-means can **group similar indents together**, enabling us to identify distinct microstructural regions or material layers in a **purely data-driven** way — without needing to know their locations beforehand.

---

## Preparation: How Many Clusters Should We Use?

### Evaluating Cluster Quality: Inertia & Silhouette Score

A fundamental challenge with k-means is choosing the right value for `k` — the number of clusters. Since k-means is unsupervised, there is no ground truth to tell us the "correct" answer. Instead, we rely on quantitative metrics to guide our choice. Two of the most common are:

#### Inertia (Within-Cluster Sum of Squares)

Inertia measures the **total squared distance from each data point to its assigned cluster centroid**. Lower inertia means tighter, more compact clusters.

- **Interpretation:**  
  Low inertia indicates that the data points are close to their centroids. However, inertia **always decreases** as you add more clusters (in the extreme case, k = n gives inertia = 0). So inertia alone cannot determine the optimal number of clusters.

- **The Elbow Method:**  
  Plot inertia as a function of `k`. The resulting curve typically shows a steep drop followed by a flattening. The "elbow" — the point where the curve bends — indicates where **adding more clusters yields diminishing returns**. This is usually a good starting point for choosing `k`.

#### Silhouette Score

The Silhouette Score measures **how similar a data point is to its own cluster compared to other clusters**. It ranges from −1 to +1:

- **Near +1:** The point fits well in its cluster and is far from neighboring clusters → good separation.  
- **Near 0:** The point sits on or near the boundary between two clusters → ambiguous assignment.  
- **Negative:** The point may be assigned to the wrong cluster → poor clustering.

- **Interpretation:**  
  A higher average Silhouette Score indicates better-defined clusters. Unlike inertia, the Silhouette Score **can decrease** if you use too many clusters, making it a more informative metric for model selection.

---

## Code: Choosing the Number of k-means Clusters

Use the inertia and Silhouette Score plots below to determine **how many clusters work best** for the k-means analysis.  
If the results are unclear, go back and **adjust the PCA settings or the feature set** to see how the metrics change.

In [ ]:
#=== Change from here ===#
features_source = "PCA" #<-- "PCA" or "features"
max_number_cluster = 10
#========================#

#=== Dont change from here ===#
silhouette_scores = []
inertias = []
ks = []

# Select data basis
if features_source == 'PCA':
    if ("PCA","PC1") not in df.columns:
        raise ValueError(f"Für 'features_source = PCA' bitte zuvor die PCA-Analyse (Code:Durchführen PCA) ausführen!")
    X = df['PCA'].dropna()
elif features_source == 'features':
    X = features.copy()
else:
    raise ValueError("features_source must be 'PCA' or 'features'.")

for k in range(2,max_number_cluster):
    kmeans = KMeans(n_clusters=k, n_init='auto', random_state=42)
    labels = kmeans.fit_predict(X)
    inertia = kmeans.inertia_
    inertias.append(inertia)
    ks.append(k)

# Silhouette Score is only defined for k > 1
    score = silhouette_score(X, labels)
    silhouette_scores.append(score)
    print(f"k = {k}, Inertia = {inertia:.2f}, Silhouette Score = {score:.3f}")

# Plotting
fig, ax1 = plt.subplots(figsize=(9, 5))

color1 = 'tab:blue'
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia (Elbow)', color=color1)
ax1.plot(ks, inertias, marker='o', color=color1, label='Inertia')
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()  # second y-axis for Silhouette Score
color2 = 'tab:green'
ax2.set_ylabel('Silhouette Score', color=color2)
ax2.plot(ks, silhouette_scores, marker='s', linestyle='--', color=color2, label='Silhouette Score')
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('KMeans: Elbow Plot & Silhouette Score')
fig.tight_layout()
plt.grid(True)
plt.show()


## Code: k-means Clustering

Use the metrics from the previous step to make an informed decision about the number of clusters.  
If needed, adjust the feature set or the number of PCA components (_go back to the PCA section, change the values, and re-run the code in order from there to here_). Observe how the cluster quality changes.

Unlike the exploratory analysis above, you are now choosing a **fixed** number of clusters. Running this cell adds a new column to the DataFrame that assigns each indent to a k-means cluster label.

In [ ]:
# === change parameter from here ===
use_pca = False  # True = using PCA axis for Clustering, False = using orignal feature set 
n_clusters = 9  # number of clusters
#=========================#

#=== dont change paramter from here ===#

if use_pca:
    # extract PCA columns from MultiIndex
    X_kmeans = df['PCA'].dropna()
else:
    # using the feature set defind early as "features = ..." 
    X_kmeans = features.copy()

# apllying index columns to the KMeans DataFrame
X_kmeans = X_kmeans.copy()
X_kmeans.index.name = df.index.name

# KMeans calculations 
kmeans = KMeans(n_clusters=n_clusters, n_init='auto', random_state=42)
labels = kmeans.fit_predict(X_kmeans)

# using strings for labelings instaed of numbers
label_names = [f"Cluster {i+1}" for i in labels]
labels_df = pd.DataFrame(label_names, index=X_kmeans.index)

# creating MultiIndex colum name for joining labels_df with df
labels_df.columns = pd.MultiIndex.from_tuples([('KMeans', 'Label')])

# deleting KMeans grouping if it already exists
if ('KMeans', 'Label') in df.columns:
    df = df.drop(columns=('KMeans', 'Label'))

# join KMeans Label_DataFrame with the original Dataframe df
df = df.join(labels_df)

display(df)

# Clustering with Gaussian Mixture Models (GMM)

## What Is GMM?

Gaussian Mixture Models (GMM) are a **probabilistic** clustering method and can be thought of as a more flexible generalization of k-means. Instead of making a hard assignment of each data point to exactly one cluster, GMM assumes the data was generated by a **mixture of several multivariate Gaussian (normal) distributions**. Each data point is assigned a **probability of belonging to each cluster** — this is known as **soft (or probabilistic) clustering**.

This flexibility allows GMM to model **clusters of different shapes, sizes, and orientations** — unlike k-means, which implicitly assumes that all clusters are spherical and roughly the same size.

The algorithm uses the **Expectation-Maximization (EM)** framework and proceeds iteratively:

1. **Initialization:**  
   Start with `k` Gaussian components, each with initial values for its mean, covariance matrix, and mixing weight (the fraction of data it represents).

2. **E-Step (Expectation):**  
   Compute the **posterior probability** (or "responsibility") that each data point belongs to each Gaussian component.

3. **M-Step (Maximization):**  
   Update each Gaussian's parameters (mean, covariance, mixing weight) to maximize the likelihood of the data given the current responsibilities.

4. **Iterate:**  
   Repeat the E- and M-steps until the parameters converge (i.e., stop changing significantly) or a stopping criterion is met.

Because GMM uses full covariance modeling, it can capture **elliptical and overlapping clusters** — a significant advantage over k-means when the true groupings in the data are not well-separated or not spherical.

---

## Model Selection with Log-Likelihood

Because GMM is a probabilistic model, its quality is evaluated using the **log-likelihood** rather than distance-based metrics like inertia. The log-likelihood measures **how probable the observed data is under the fitted model**.

- **Higher log-likelihood = better fit to the data.**
- For easier comparison with k-means, we often plot the **negative log-likelihood** as a function of the number of components. This produces a curve similar to the elbow plot, where you look for a bend indicating diminishing returns.

---

## When Should You Use GMM Over k-means?

GMM is particularly useful when:

- **Clusters have different shapes, sizes, or densities** (i.e., they are not all spherical)
- **Clusters overlap**, and a hard assignment would lose important nuance
- You need **probabilities of cluster membership** (e.g., to flag uncertain or mixed-phase indents)
- **Material phases or microstructural regions transition gradually** rather than having sharp boundaries

---

## Code: Choosing the Number of GMM Components
Run the code below and use the Silhouette Score and the elbow plot to determine the most appropriate number of mixture components for the GMM, given your current feature set and PCA configuration. Adjust the PCA settings if needed. Follow the same approach you used for k-means.

In [ ]:
# === Change from here === #
features_source = "PCA"  # <-- "PCA" or "features"
max_number_cluster = 10
# ======================== #

# === Don't change from here === #
silhouette_scores = []
neg_log_likelihoods = []
ks = []

# Select data basis
if features_source == 'PCA':
    X = df['PCA'].dropna()
elif features_source == 'features':
    X = features.copy()
else:
    raise ValueError("features_source must be 'PCA' or 'features'.")

for k in range(2, max_number_cluster):
    gmm = GaussianMixture(n_components=k, random_state=42, n_init=5)
    gmm.fit(X)
    labels = gmm.predict(X)
    
    # Log-likelihood (higher is better, so we invert it for Elbow-like plot)
    neg_ll = -gmm.score(X) * len(X)
    neg_log_likelihoods.append(neg_ll)
    ks.append(k)

    # Silhouette Score
    score = silhouette_score(X, labels)
    silhouette_scores.append(score)
    print(f"k = {k}, -LogLikelihood = {neg_ll:.2f}, Silhouette Score = {score:.3f}")

# Plotting
fig, ax1 = plt.subplots(figsize=(9, 5))

color1 = 'tab:purple'
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('− Log-Likelihood (Elbow)', color=color1)
ax1.plot(ks, neg_log_likelihoods, marker='o', color=color1, label='− Log-Likelihood')
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()  # second y-axis for Silhouette Score
color2 = 'tab:green'
ax2.set_ylabel('Silhouette Score', color=color2)
ax2.plot(ks, silhouette_scores, marker='s', linestyle='--', color=color2, label='Silhouette Score')
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('GMM: Log-Likelihood & Silhouette Score')
fig.tight_layout()
plt.grid(True)
plt.show()


## Code: GMM Clustering

Use the Silhouette Score and elbow plot from the previous step to choose the optimal number of clusters for GMM.

If needed:

- adjust the **feature set**, or  
- change the number of **PCA components**

(go back to the PCA section, update the values, and re-run the code from there to this point).

Observe how the clustering quality changes.

You are now choosing a fixed number of mixture components. Running this cell adds a new column to the DataFrame that assigns each indent a GMM cluster label.

In [ ]:
# === change parameter from here === #
use_pca = False   # True = using PCA axis for Clustering, False = using original feature set 
n_clusters = 4   # number of mixture components
gmm_n_init = 10   # number of initializations for GMM
# ======================== #

# === don't change parameter from here === #

if use_pca:
    # extract PCA columns from MultiIndex
    X_gmm = df['PCA'].dropna()
else:
    # using the feature set defined earlier as "features = ..."
    X_gmm = features.copy()

# apply index columns to the GMM DataFrame
X_gmm = X_gmm.copy()
X_gmm.index.name = df.index.name

# GMM fitting
gmm = GaussianMixture(n_components=n_clusters, random_state=42, n_init=gmm_n_init)
gmm.fit(X_gmm)
labels = gmm.predict(X_gmm)
probs = gmm.predict_proba(X_gmm)  # shape: (n_samples, n_components)

# using strings for labelings instead of numbers
label_names = [f"Cluster {i+1}" for i in labels]
labels_df = pd.DataFrame(label_names, index=X_gmm.index)

# create MultiIndex column name for joining labels_df with df
labels_df.columns = pd.MultiIndex.from_tuples([('GMM', 'Label')])

# create a DataFrame for component probabilities with MultiIndex columns
prob_cols = [ ( 'GMM', f'comp_{i+1}' ) for i in range(n_clusters) ]
prob_df = pd.DataFrame(probs, index=X_gmm.index, columns=pd.MultiIndex.from_tuples(prob_cols))

# delete existing GMM grouping/prob columns if they already exist
cols_to_drop = []
if ('GMM', 'Label') in df.columns:
    cols_to_drop.append(('GMM', 'Label'))
for col in prob_df.columns:
    if col in df.columns:
        cols_to_drop.append(col)
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)

# join label and probability DataFrames with the original DataFrame df
df = df.join(labels_df)
df = df.join(prob_df)

display(df)


# Data Preprocessing

If you look at the Silhouette Scores and elbow plots for both k-means and GMM, you will likely notice that for most feature sets and PCA configurations, there is **no clear elbow** in the inertia or negative log-likelihood curves. On top of that, the Silhouette Score typically stays **below 0.4**, indicating poor cluster separation.

There are two common reasons for this:

- The dataset is too small (not enough indents).  
- There is too much variability (noise) within the data.  

Whether it is better to have more data with higher scatter, or less data with lower scatter, depends on the specific dataset and must be determined empirically.

Making this judgment requires a solid understanding of the data:

- How were the measurements performed?  
- What do the measured quantities physically represent?

Load the file **"DataWithAdditional_Infos.csv"** and explore the expanded dataset.  
All reported mean values ("mean") and standard deviations ("std") were computed over an indentation depth window of **200 to 400 nm**. The full force–depth, modulus–depth, and hardness–depth curves are not included in this exercise due to their size.

Over that same depth range (200–400 nm), a **linear fit** was applied to the hardness–depth, modulus–depth, and stiffness–depth data. The slope of this fit indicates whether each quantity is increasing, decreasing, or staying constant over the evaluation window.  
At a sufficiently large indentation depth, these values should plateau. If they do not, this may indicate that:

- an indent landed on a **phase boundary** and sampled two phases at once, or  
- multiple phases are **layered on top of each other** within the first 400 nm.

These indents blur the boundaries between clusters and can significantly degrade clustering performance. Removing them from the dataset can improve cluster quality.

---

## Code: Data Filtering

Use the code below to filter the data and **remove all indents whose elastic modulus varies by more than 5%** across the evaluation depth range.  
The relevant column is: **("MODULUS GPa", "Slope%")**.

You can run this cell multiple times with different settings — change:

- the `value_to_filter` parameter  
- the `min_value` and `max_value` thresholds

to apply multiple filtering criteria.

After filtering, **re-run all code cells starting from the PCA section** and check whether the elbow plots and Silhouette Scores for k-means and GMM have improved.

In [ ]:
# === change parameter from here === #
value_to_filter = ("MODULUS GPa","Slope%")
min_value = -5
max_value = 5
# ======================== #

# === don't change parameter from here === #

if value_to_filter not in df.columns:
    raise ValueError(f"Die Spalte {value_to_filter} befindet sich nicht in der Tabelle. Überprüfen Sie, ob der richtige Datensatz eingeladen ist.\n Laden Sie den richtige Datensatz ein und und führen Sie nochmals die KMeans und GMM Analyse durch.")
#image size
df = df[(df[value_to_filter] <= max_value) &(df[value_to_filter] >= min_value)]

#only to show that the length of the DataFrame has changed
display(df[("MODULUS GPa","Slope%")])